# Hipóteses de NPS - Safira

Implementação computacional das hipóteses descritas na seção 4.2.4 (`Hipóteses`) de
`documents/documentacao.md`:

- **Hipótese 1**: uma tripulação mal avaliada pode pesar tanto quanto um atraso grave
- **Hipótese 2**: o canal de compra revela um perfil de cliente com sensibilidades diferentes
- **Hipótese 3**: cancelamentos repentinos geram mais detratores
- **Hipótese 4**: o detrator crônico
- **Hipótese 5**: o Cliente mais fidelizado é o menos tolerante à falha operacional
- **Hipótese 6**: a fragmentação da jornada eleva a detração por exposição, não por desgaste

A base recomendada para esta análise é `data/processed/base_analitica.parquet`, o Parquet
tratado que `notebooks/pre-processamento.ipynb` gera a partir da integração de NPS, perfil
do Cliente e informação de viagem (ver `scripts/preprocessamento_nps.py`, disponível na
branch `develop`). Ela não é versionada neste repositório (ver `.gitignore` e a Política de
Privacidade da seção 4.1.8). Aponte `DATA_PATH`, abaixo, para o local desse arquivo na sua
máquina ou no Google Drive quando este notebook for aberto no Colab; apontar para uma base
bruta ou intermediária, em vez da base tratada, serve apenas para diagnóstico ou reprodução
de uma etapa específica do pré-processamento, não para a análise oficial das hipóteses.

**Correção em relação à versão anterior:** a primeira versão deste notebook rodava contra
uma base diferente, sem as colunas derivadas no pré-processamento (`ATRASO_CHEGADA`,
`CANAL_COMPRA`, `ID_GOLDENRECORD`, `TIER_VIAGEM`), e por isso registrava avisos de "coluna
não documentada". A base analítica atual já traz essas colunas prontas, então esta versão
recalcula as hipóteses do zero sobre os dados corretos, em vez de reaproveitar os números
da versão anterior.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

## Configuração

Ajuste `DATA_PATH` se o arquivo estiver em outro local. O dicionário `COL` isola os nomes
de coluna num único lugar para que qualquer ajuste futuro não exija editar cada hipótese.

**`COL` é local a este notebook, de forma proposital.** Ele existe só para as hipóteses da
seção 4.2.4 e não é compartilhado com `notebooks/pre-processamento.ipynb`,
`notebooks/escalonamento_anexo_a1.ipynb` ou `scripts/preprocessamento_nps.py`, que não o
importam nem dependem dele. Se um nome de coluna mudar na base analítica, o ajuste é feito
aqui, sem afetar o pré-processamento; não é uma configuração compartilhada entre as etapas.

In [ ]:
# DATA_PATH deve apontar para a base analítica TRATADA, isto é, o Parquet gerado por
# notebooks/pre-processamento.ipynb (não para os arquivos brutos de data/raw nem para uma
# exportação intermediária). O caminho varia por máquina e por ambiente, então ajuste-o
# aqui antes de rodar; os valores abaixo são só exemplos de onde esse arquivo costuma ficar:
#   - Localmente, junto do restante do projeto:
#       DATA_PATH = "../data/processed/base_analitica.parquet"
#   - No Google Colab, após montar o Drive:
#       from google.colab import drive
#       drive.mount("/content/drive")
#       DATA_PATH = "/content/drive/MyDrive/<caminho-do-projeto>/data/processed/base_analitica.parquet"
#
# Apontar para uma base bruta ou para uma etapa intermediária do pré-processamento é útil
# apenas para diagnóstico ou para reproduzir um passo específico da limpeza; os números
# publicados na seção 4.2.4 assumem a base tratada.
DATA_PATH = "../data/processed/base_analitica.parquet"

# Nomes de coluna da base analítica (saída do pré-processamento). ATRASO_CHEGADA,
# CANAL_COMPRA, ID_GOLDENRECORD e TIER_VIAGEM são colunas derivadas na integração das
# tabelas de NPS, perfil do Cliente e informação de viagem; não aparecem no dicionário
# de dados bruto da seção 4.1.3, que documenta apenas a base de pesquisa isolada.
#
# Este dicionário é interno a este notebook: centraliza os nomes de coluna usados pelas
# hipóteses e facilita a manutenção da análise, mas não é uma configuração compartilhada
# com o pré-processamento. Os demais notebooks e o script de pré-processamento não o
# utilizam, então mudanças aqui não precisam (e não devem) ser refletidas lá.
COL = {
    "target": "NPS_PRINCIPAL",                  # -100 Detrator, 0 Neutro, 100 Promotor
    "comissarios": "NPS_COMISSARIOS",
    "pilotos": "NPS_PILOTOS",
    "atraso_saida": "ESTATISTICA_ATRASOSAIDA",  # atraso na PARTIDA, em minutos
    "atraso_chegada": "ATRASO_CHEGADA",         # atraso na CHEGADA, em minutos (derivado)
    "cancelamento": "CANCELAMENTO_VOO",
    "antecedencia_cancelamento": "ANTECEDENCIA_CANCELAMENTO",  # em dias, por definição da seção 4.1.3
    "canal_compra": "CANAL_COMPRA",
    "wifi": "NPS_WIFI",
    "entretenimento": "NPS_ENTRETENIMENTO",
    "embarque": "NPS_EMBARQUE",
    "cliente_id": "ID_GOLDENRECORD",            # identificador do Cliente ao longo do tempo
    "data_voo": "DATA_STD_CONVERTIDA",          # DATA_STD já convertida para datetime no pré-processamento
    "tier_viagem": "TIER_VIAGEM",
    "rota": "BASE_AIRPORTLEG",                  # sequência de aeroportos da jornada, separados por "/"
    "tempo_voo": "TEMPO_VOO",
    "respondent_id": "RESPONDENT_ID",  # desempata respostas do mesmo Cliente no mesmo dia
}

LIMIAR_NOTA_NEGATIVA = 6   # notas 0 a 6 na escala 0-10, mesmo corte de Detrator do NPS_PRINCIPAL
LIMIAR_NOTA_POSITIVA = 9   # notas 9 e 10, mesmo corte de Promotor do NPS_PRINCIPAL
LIMIAR_ATRASO_GRAVE_MIN = 120

In [ ]:
df = pd.read_parquet(DATA_PATH)
print(f"{len(df):,} registros carregados de {DATA_PATH}")

def checar_colunas(colunas_necessarias, contexto):
    # Mantido como checagem defensiva: se alguém rodar este notebook contra uma
    # exportação parcial da base analítica, o aviso aponta exatamente a coluna que
    # falta em vez de estourar um KeyError no meio da hipótese.
    faltantes = [c for c in colunas_necessarias if c not in df.columns]
    if faltantes:
        print(f"[{contexto}] colunas ausentes na base carregada: {faltantes}")
        print(f"[{contexto}] ajuste o dicionário COL para os nomes reais antes de continuar.")
    return not faltantes

# DATA_STD_CONVERTIDA já sai como datetime do pré-processamento; a conversão só roda
# se alguém apontar DATA_PATH para uma base mais antiga que ainda não passou por ele.
nome_data_voo = COL["data_voo"]
if nome_data_voo in df.columns and not pd.api.types.is_datetime64_any_dtype(df[nome_data_voo]):
    df[nome_data_voo] = pd.to_datetime(df[nome_data_voo], dayfirst=True, errors="coerce")

## Funções auxiliares

Reaproveitadas por mais de uma hipótese: a definição de Detrator, a leitura de
`CANCELAMENTO_VOO` e o recorte de "voo perfeito" (pontual e sem cancelamento) que isola
problemas de tripulação e de canal de compra de qualquer influência operacional.

In [ ]:
def to_bool(serie):
    # CANCELAMENTO_VOO já vem como bool na base analítica atual, mas a função
    # trata também o formato de texto Sim/Não: serie.astype(bool) direto é uma
    # armadilha aqui, porque qualquer string não vazia é truthy em Python, então
    # "Nao" também viraria True. Preferimos avisar sobre um valor não reconhecido a
    # deixá-lo virar True silenciosamente.
    if pd.api.types.is_bool_dtype(serie):
        return serie.fillna(False)
    if pd.api.types.is_numeric_dtype(serie):
        return serie.fillna(0).astype(bool)
    normalizada = serie.astype(str).str.strip().str.lower()
    verdadeiros = {"true", "1", "sim", "yes", "y", "verdadeiro"}
    falsos = {"false", "0", "nao", "não", "no", "n", "falso"}
    convertida = normalizada.map(lambda v: True if v in verdadeiros else (False if v in falsos else None))
    nao_reconhecidos = sorted(set(normalizada[convertida.isna() & serie.notna()]))
    if nao_reconhecidos:
        print(f"[to_bool] valores não reconhecidos em '{serie.name}', tratados como False: {nao_reconhecidos}")
    return convertida.fillna(False).astype(bool)

def is_detrator(serie_nps):
    return serie_nps.eq(-100)

def voos_pontuais_sem_cancelamento(base):
    # Isola qualquer influência operacional (atraso, cancelamento) para que o efeito
    # medido em cada hipótese venha só da variável de interesse (tripulação, canal etc.).
    pontual = base[COL["atraso_saida"]] <= 0
    sem_cancelamento = ~to_bool(base[COL["cancelamento"]])
    return base[pontual & sem_cancelamento].copy()

def taxa_detratores(base, coluna_grupo=None):
    alvo = is_detrator(base[COL["target"]])
    if coluna_grupo is None:
        return alvo.mean()
    return base.assign(_detrator=alvo).groupby(coluna_grupo, observed=True)["_detrator"].mean()

## Hipótese 1: uma tripulação mal avaliada pode pesar tanto quanto um atraso grave

Isola voos pontuais e sem cancelamento, compara a taxa de detratores entre avaliações
negativas e positivas de comissários e pilotos, e contrasta com a taxa de detratores no
pior cenário puramente operacional da base (atraso na chegada acima de 120 minutos).

In [ ]:
if checar_colunas(
    [COL["comissarios"], COL["pilotos"], COL["atraso_saida"], COL["cancelamento"],
     COL["atraso_chegada"], COL["target"]],
    "Hipótese 1",
):
    base_limpa = voos_pontuais_sem_cancelamento(df)
    print(f"Voos pontuais e sem cancelamento: {len(base_limpa):,} de {len(df):,}")

    def taxa_por_avaliacao(base, coluna_nota):
        # Compara os dois extremos da escala (0-6 vs. 9-10) em vez de correlacionar a
        # nota diretamente com a detração: é a mesma leitura da tabela de detratores
        # do NPS_PRINCIPAL, aplicada a uma sub-nota da jornada.
        notas = base[coluna_nota].dropna()
        negativa = base.loc[notas[notas <= LIMIAR_NOTA_NEGATIVA].index]
        positiva = base.loc[notas[notas >= LIMIAR_NOTA_POSITIVA].index]
        return taxa_detratores(negativa), taxa_detratores(positiva), len(negativa), len(positiva)

    for rotulo, coluna in [("Comissários", COL["comissarios"]), ("Pilotos", COL["pilotos"])]:
        neg, pos, n_neg, n_pos = taxa_por_avaliacao(base_limpa, coluna)
        print(f"{rotulo}: avaliação negativa {neg:.1%} (n={n_neg:,}), "
              f"avaliação positiva {pos:.1%} (n={n_pos:,})")

    # O contraponto usa a base inteira (não só voos pontuais): queremos o pior cenário
    # operacional que a malha realmente produz, não um recorte que já excluiria atraso.
    atraso_grave = df[df[COL["atraso_chegada"]] > LIMIAR_ATRASO_GRAVE_MIN]
    print(f"\nAtraso na chegada acima de {LIMIAR_ATRASO_GRAVE_MIN} min "
          f"({len(atraso_grave):,} voos): {taxa_detratores(atraso_grave):.1%} de detratores")

## Hipótese 2: o canal de compra revela um perfil de cliente com sensibilidades diferentes

Mesma base de voos pontuais e sem cancelamento. Primeiro compara a taxa de detratores por
canal de compra; depois, dentro de cada canal, mede o quanto uma avaliação negativa de
wifi/entretenimento e de comissários/embarque eleva a taxa de detratores em relação a uma
avaliação positiva.

In [ ]:
if checar_colunas(
    [COL["canal_compra"], COL["atraso_saida"], COL["cancelamento"], COL["target"]],
    "Hipótese 2",
):
    base_limpa = voos_pontuais_sem_cancelamento(df)
    taxa_por_canal = taxa_detratores(base_limpa, COL["canal_compra"]).sort_values()
    print("Taxa de detratores por canal de compra (voos pontuais e sem cancelamento):")
    print(taxa_por_canal.apply(lambda v: f"{v:.1%}"))

    def efeito_avaliacao_negativa(base, coluna_nota, canal):
        # Diferença de taxa (negativo menos positivo) em vez de razão: com canais de
        # baixo volume (CALLCENTER, AEROPORTO, OTHER) uma razão amplificaria ruído de
        # amostra pequena; a diferença em pontos percentuais é mais estável para comparar
        # entre canais com "n" tão desiguais.
        base_canal = base[base[COL["canal_compra"]] == canal]
        notas = base_canal[coluna_nota].dropna()
        negativa = base_canal.loc[notas[notas <= LIMIAR_NOTA_NEGATIVA].index]
        positiva = base_canal.loc[notas[notas >= LIMIAR_NOTA_POSITIVA].index]
        if len(negativa) == 0 or len(positiva) == 0:
            return np.nan
        return taxa_detratores(negativa) - taxa_detratores(positiva)

    for canal in taxa_por_canal.index:
        print(f"\n{canal}:")
        for rotulo, coluna in [
            ("wifi", COL["wifi"]),
            ("entretenimento", COL["entretenimento"]),
            ("comissários", COL["comissarios"]),
            ("embarque", COL["embarque"]),
        ]:
            efeito = efeito_avaliacao_negativa(base_limpa, coluna, canal)
            if pd.notna(efeito):
                print(f"  variação na detração ({rotulo} negativo vs. positivo): {efeito:+.1%}")
            else:
                print(f"  {rotulo}: dados insuficientes nesse canal")

## Hipótese 3: cancelamentos repentinos geram mais detratores

Testa se a taxa de detratores entre voos cancelados cai conforme aumenta a antecedência
com que o Cliente foi avisado do cancelamento (`ANTECEDENCIA_CANCELAMENTO`, documentada na
seção 4.1.3 como o intervalo entre o cancelamento e a partida originalmente prevista, em
dias). Ao final, compara a faixa de maior antecedência com a taxa de detratores dos voos
que nunca foram cancelados.

In [ ]:
if checar_colunas(
    [COL["antecedencia_cancelamento"], COL["cancelamento"], COL["target"]],
    "Hipótese 3",
):
    cancelados = df[to_bool(df[COL["cancelamento"]])].copy()
    cancelados["_detrator"] = is_detrator(cancelados[COL["target"]])
    print(f"Voos cancelados: {len(cancelados):,} de {len(df):,}")

    # Faixas em dias, por sequência de eventos plausível para o Cliente reagir: aviso
    # no mesmo dia, até uma semana, até um mês, e acima de um mês. O corte de 6 dias
    # (e não 7) é proposital: separa "menos de uma semana" de "1 a 6 dias", evitando que
    # o limite caia exatamente sobre o valor mais comum de antecedência de uma semana.
    LIMIARES_ANTECEDENCIA = [-1, 0, 6, 24, 48, np.inf]
    ROTULOS_ANTECEDENCIA = [
        "0 dias (aviso no mesmo dia)", "1 a 6 dias", "7 a 24 dias",
        "25 a 48 dias", "mais de 48 dias",
    ]
    cancelados["_faixa_antecedencia"] = pd.cut(
        cancelados[COL["antecedencia_cancelamento"]],
        bins=LIMIARES_ANTECEDENCIA, labels=ROTULOS_ANTECEDENCIA,
    )

    tabela_antecedencia = cancelados.groupby("_faixa_antecedencia", observed=True)["_detrator"].agg(
        taxa_detratores="mean", n="size"
    )
    print("\nTaxa de detratores por antecedência do aviso de cancelamento:")
    print(tabela_antecedencia)

    tabela_contingencia = pd.crosstab(cancelados["_faixa_antecedencia"], cancelados["_detrator"])
    chi2, p_valor, gl, _ = stats.chi2_contingency(tabela_contingencia)
    print(f"\nqui-quadrado = {chi2:.1f}; gl = {gl}; "
          f"p {'< 0,001' if p_valor < 0.001 else f'= {p_valor:.3g}'}")

    taxa_sem_cancelamento = taxa_detratores(df[~to_bool(df[COL["cancelamento"]])])
    print(f"\nTaxa de detratores em voos sem cancelamento (referência): {taxa_sem_cancelamento:.1%}")

## Hipótese 4: o detrator crônico

Requer um identificador de Cliente estável entre respostas (`COL["cliente_id"]`) para
ordenar o histórico por `COL["data_voo"]` e comparar cada resposta com a resposta anterior
do mesmo Cliente. Só são removidos os registros sem `ID_GOLDENRECORD`; uma resposta sem data
continua na análise e é ordenada ao final do histórico do seu Cliente, e o `RESPONDENT_ID`
desempata respostas do mesmo dia de forma determinística. Reproduz os três
testes do quadro da seção 4.2.4 (concentração com qui-quadrado 2x2, predição por taxas
condicionais e resistência a controles operacionais progressivos) além da regressão
logística final.

In [ ]:
if checar_colunas(
    [COL["cliente_id"], COL["data_voo"], COL["respondent_id"], COL["target"],
     COL["atraso_saida"], COL["cancelamento"]],
    "Hipótese 4",
):
    # Só removemos quem não tem ID_GOLDENRECORD: sem ele não há como saber a quem a
    # resposta pertence. Uma resposta sem data continua na análise; o groupby().shift(1)
    # ordena por [Cliente, data, RESPONDENT_ID], e o pandas manda NaT para o fim de cada
    # grupo, então essa resposta só pode aparecer como a mais recente do seu Cliente.
    historico = df.dropna(subset=[COL["cliente_id"]]).sort_values(
        [COL["cliente_id"], COL["data_voo"], COL["respondent_id"]]
    ).copy()
    historico["_detrator"] = is_detrator(historico[COL["target"]])
    # Voo perfeito exige atraso zero na saída e na chegada, e ausência de
    # cancelamento: o corte B verifica as duas medidas de atraso.
    historico["_pontual"] = (
        (historico[COL["atraso_saida"]] <= 0)
        & (historico[COL["atraso_chegada"]] <= 0)
        & (~to_bool(historico[COL["cancelamento"]]))
    )
    historico["_detrator_anterior"] = historico.groupby(COL["cliente_id"])["_detrator"].shift(1)
    # O corte C só exige que a CHEGADA do voo anterior tenha sido pontual, não que
    # o voo anterior inteiro fosse perfeito; essa é a condição definida para este corte.
    historico["_atraso_chegada_anterior"] = historico.groupby(COL["cliente_id"])[COL["atraso_chegada"]].shift(1)

    respostas_com_historico = historico[historico["_detrator_anterior"].notna()].copy()
    respostas_com_historico["_detrator_anterior"] = respostas_com_historico["_detrator_anterior"].astype(bool)

    print(
        f"Respostas com pelo menos uma resposta anterior do mesmo Cliente: "
        f"{len(respostas_com_historico):,} de {len(historico):,} "
        f"({len(respostas_com_historico) / len(historico):.1%})"
    )

In [ ]:
# Teste 1: concentração, entre Clientes com exatamente duas respostas
contagem_por_cliente = historico.groupby(COL["cliente_id"]).size()
clientes_com_duas = contagem_por_cliente[contagem_por_cliente == 2].index
duplas = historico[historico[COL["cliente_id"]].isin(clientes_com_duas)]

primeira = duplas.groupby(COL["cliente_id"])["_detrator"].first()
segunda = duplas.groupby(COL["cliente_id"])["_detrator"].last()
tabela_2x2 = pd.crosstab(primeira, segunda)

chi2, p_valor, gl, esperado = stats.chi2_contingency(tabela_2x2, correction=False)
esperado_df = pd.DataFrame(esperado, index=tabela_2x2.index, columns=tabela_2x2.columns)

print(f"Clientes com exatamente duas respostas: {len(clientes_com_duas):,}")
print("\nTabela observada (linha = primeira resposta, coluna = segunda resposta):")
print(tabela_2x2)
print("\nTabela esperada sob independência:")
print(esperado_df.round(1))
print(f"\nqui-quadrado = {chi2:.1f}; gl = {gl}; p = {p_valor:.3g}")

In [ ]:
# Teste 2: predição
taxa_pos = respostas_com_historico.loc[respostas_com_historico["_detrator_anterior"], "_detrator"].mean()
taxa_neg = respostas_com_historico.loc[~respostas_com_historico["_detrator_anterior"], "_detrator"].mean()

print(f"Taxa de detração | detratou na resposta anterior: {taxa_pos:.1%}")
print(f"Taxa de detração | não detratou na resposta anterior: {taxa_neg:.1%}")
print(f"Razão: {taxa_pos / taxa_neg:.1f}x")

# Teste 3: resistência a controles operacionais progressivos
def razao_detratou_vs_nao(base):
    pos = base.loc[base["_detrator_anterior"], "_detrator"].mean()
    neg = base.loc[~base["_detrator_anterior"], "_detrator"].mean()
    return pos, neg, pos / neg

voo_atual_perfeito = respostas_com_historico[respostas_com_historico["_pontual"]]
corte_c = respostas_com_historico[
    respostas_com_historico["_pontual"]
    & (respostas_com_historico["_atraso_chegada_anterior"] <= 0)
]

# Se a razão caísse ao remover as causas operacionais, o efeito seria explicado pelo
# voo, não pela pessoa; o teste é justamente ver se ela sobe em vez de cair.
print()
for rotulo, base in [
    ("A. Todas as respostas com histórico", respostas_com_historico),
    ("B. Voo atual sem atraso na saída, sem atraso na chegada e sem cancelamento", voo_atual_perfeito),
    ("C. B, exigindo também que o voo anterior tenha chegado sem atraso", corte_c),
]:
    pos, neg, razao = razao_detratou_vs_nao(base)
    print(f"{rotulo} (n={len(base):,}): {razao:.1f}x ({pos:.1%} vs. {neg:.1%})")

In [ ]:
# Regressão logística: detrator ~ detratou_antes + atraso + cancelamento
# Controla pelas duas principais falhas operacionais para isolar o efeito do histórico
# do Cliente do efeito de o voo atual simplesmente ter sido pior.
modelo_dados = respostas_com_historico.dropna(subset=[COL["atraso_chegada"], COL["cancelamento"]]).copy()
modelo_dados["_cancelado"] = to_bool(modelo_dados[COL["cancelamento"]]).astype(int)
modelo_dados["_detrator_anterior_int"] = modelo_dados["_detrator_anterior"].astype(int)
modelo_dados["_detrator_int"] = modelo_dados["_detrator"].astype(int)

X = sm.add_constant(modelo_dados[["_detrator_anterior_int", COL["atraso_chegada"], "_cancelado"]])
y = modelo_dados["_detrator_int"]
modelo_logit = sm.Logit(y, X).fit(disp=False)
print(modelo_logit.summary2().tables[1])

coef = modelo_logit.params["_detrator_anterior_int"]
ic_inf, ic_sup = modelo_logit.conf_int().loc["_detrator_anterior_int"]
p_valor_coef = modelo_logit.pvalues["_detrator_anterior_int"]

print(
    f"\nOdds ratio (detratou na resposta anterior): {np.exp(coef):.2f} "
    f"[IC 95%: {np.exp(ic_inf):.2f} - {np.exp(ic_sup):.2f}], p = {p_valor_coef:.3g}"
)

## Hipótese 5: o Cliente mais fidelizado é o menos tolerante à falha operacional

Testa se o efeito do atraso na partida sobre a taxa de detratores muda conforme o tier
de fidelidade do Cliente (`COL["tier_viagem"]`). O atraso é discretizado na mesma
taxonomia da seção 4.2.1 (`_faixa_atraso`: sem atraso, 15-60 min, 61-120 min, mais de
120 min).

- **H0**: a taxa de detratores por faixa de atraso é independente do tier de fidelidade
  (tier e faixa de atraso agem apenas de forma aditiva sobre a detração).
- **H1**: existe interação, e o efeito do atraso sobre a detração é maior nos tiers mais
  altos.
- **alfa = 0,05**

In [ ]:
if checar_colunas(
    [COL["tier_viagem"], COL["atraso_saida"], COL["target"]],
    "Hipótese 5",
):
    ORD_ATRASO_H5 = ["a. Sem Atraso", "b. 15m - 60m", "c. 61m - 120m", "d. Mais de 120m"]

    def faixa_atraso(minutos):
        # Mesma discretização da seção 4.2.1: bandas fechadas até 120 min e uma
        # faixa aberta acima disso.
        banda = np.select(
            [minutos < 15, minutos <= 60, minutos <= 120],
            ORD_ATRASO_H5[:3],
            default=ORD_ATRASO_H5[3],
        )
        return pd.Categorical(banda, categories=ORD_ATRASO_H5, ordered=True)

    base_h5 = df.dropna(subset=[COL["tier_viagem"], COL["atraso_saida"], COL["target"]]).copy()
    base_h5["_faixa_atraso"] = faixa_atraso(base_h5[COL["atraso_saida"]])
    base_h5["_detrator"] = is_detrator(base_h5[COL["target"]]).astype(int)

    tabela_n = pd.crosstab(base_h5[COL["tier_viagem"]], base_h5["_faixa_atraso"])
    tabela_taxa = pd.crosstab(
        base_h5[COL["tier_viagem"]], base_h5["_faixa_atraso"],
        values=base_h5["_detrator"], aggfunc="mean",
    )

    print(
        f"Respostas com tier e faixa de atraso definidos: {len(base_h5):,} de {len(df):,} "
        f"({len(base_h5) / len(df):.1%})"
    )
    print("\nTamanho de cada célula tier x faixa de atraso:")
    print(tabela_n)

    celulas_pequenas = int((tabela_n < 30).sum().sum())
    if celulas_pequenas:
        print(
            f"\n[aviso] {celulas_pequenas} célula(s) tier x faixa de atraso com menos de "
            "30 respostas: a estimativa de interação nessas células é instável."
        )

    print("\nTaxa de detratores (%) por tier e faixa de atraso:")
    print(tabela_taxa.mul(100).round(1))

### Teste de interação: razão de verossimilhanças entre modelos logísticos

Um qui-quadrado de independência na tabela cruzada testaria se tier e faixa de atraso se
associam ao *nível* de detração, mas não isola o efeito de *interação* de cada variável
isoladamente. Por isso o teste compara dois modelos logísticos aninhados: um só com
efeitos aditivos (`tier + faixa`, consistente com H0) e um com o termo de interação
explícito (`tier * faixa`, consistente com H1). A estatística de razão de
verossimilhanças segue qui-quadrado com graus de liberdade iguais à diferença de
parâmetros entre os dois modelos.

**Pressupostos verificados:** cada linha é uma resposta independente (mesmo tratamento
de granularidade das demais hipóteses deste notebook); as células pequenas ou com taxa de
detratores em 0% ou 100%, sinal de possível separação perfeita, foram sinalizadas acima e
abaixo antes do ajuste.

In [ ]:
celulas_extremas = tabela_taxa.stack()
celulas_extremas = celulas_extremas[(celulas_extremas == 0) | (celulas_extremas == 1)]
if len(celulas_extremas):
    print(
        f"[aviso] {len(celulas_extremas)} célula(s) com taxa de detratores 0% ou 100%: "
        "risco de separação perfeita no modelo de interação."
    )

ALPHA_H5 = 0.05
formula_col = COL["tier_viagem"]
modelo_aditivo = smf.logit(
    f"_detrator ~ C({formula_col}) + C(_faixa_atraso)", data=base_h5
).fit(disp=False, maxiter=200)
modelo_interacao = smf.logit(
    f"_detrator ~ C({formula_col}) * C(_faixa_atraso)", data=base_h5
).fit(disp=False, maxiter=200)

lr_estatistica = 2 * (modelo_interacao.llf - modelo_aditivo.llf)
gl_lr = modelo_interacao.df_model - modelo_aditivo.df_model
p_valor_lr = stats.chi2.sf(lr_estatistica, gl_lr)
decisao = "rejeita H0" if p_valor_lr < ALPHA_H5 else "não rejeita H0"

print("Teste de razão de verossimilhanças (interação tier x faixa de atraso):")
print(
    f"  estatística = {lr_estatistica:.2f}; gl = {gl_lr}; p = {p_valor_lr:.3g}; "
    f"alfa = {ALPHA_H5}"
)
print(f"  decisão: {decisao}")

if p_valor_lr < ALPHA_H5:
    print(
        "\nConclusão: o efeito do atraso sobre a detração depende do tier de "
        "fidelidade (interação estatisticamente significativa). Isso sustenta "
        "priorizar a atuação preventiva nos tiers mais altos quando o atraso "
        "ultrapassa as faixas de maior risco."
    )
else:
    print(
        "\nConclusão: não há evidência, nesta amostra, de que o efeito do atraso "
        "sobre a detração varie por tier de fidelidade; um modelo aditivo é "
        "suficiente."
    )

### Verificação de robustez: excluindo o tier de menor volume

`AZUL ONE` tem apenas 189 respostas na base inteira, e a célula "mais de 120 min" desse
tier tem só 2 respostas, ambas sem detração. Uma célula assim pode tanto inflar quanto
mascarar a estatística de interação por acaso de amostra pequena. O teste abaixo repete o
ajuste sem esse tier, para checar se a decisão sobre H0 depende dele.

In [ ]:
base_h5_sem_azulone = base_h5[base_h5[formula_col] != "AZUL ONE"].copy()
base_h5_sem_azulone[formula_col] = base_h5_sem_azulone[formula_col].astype(str)

modelo_aditivo_robustez = smf.logit(
    f"_detrator ~ C({formula_col}) + C(_faixa_atraso)", data=base_h5_sem_azulone
).fit(disp=False, maxiter=200)
modelo_interacao_robustez = smf.logit(
    f"_detrator ~ C({formula_col}) * C(_faixa_atraso)", data=base_h5_sem_azulone
).fit(disp=False, maxiter=200)

lr_robustez = 2 * (modelo_interacao_robustez.llf - modelo_aditivo_robustez.llf)
gl_robustez = modelo_interacao_robustez.df_model - modelo_aditivo_robustez.df_model
p_robustez = stats.chi2.sf(lr_robustez, gl_robustez)

print(f"Sem AZUL ONE (n={len(base_h5_sem_azulone):,}): estatística = {lr_robustez:.2f}; "
      f"gl = {gl_robustez}; p = {p_robustez:.3g}")
print("A decisão sobre H0 não muda ao remover o tier de menor volume."
      if (p_robustez < ALPHA_H5) == (p_valor_lr < ALPHA_H5)
      else "A decisão sobre H0 muda ao remover o tier de menor volume: investigar antes de reportar.")

### Teste mais direcionado: a hipótese compara extremos de fidelidade, não os sete tiers entre si

O teste acima (`tier * faixa_atraso`) tem 18 graus de liberdade porque testa toda
combinação entre os sete tiers e as quatro faixas, incluindo contrastes que a hipótese
não afirma, por exemplo se `TOPAZIO` reage de forma diferente de `SAFIRA`. Isso dilui a
potência do teste bem além do que a hipótese realmente precisa: ela afirma que o Cliente
**mais** fidelizado reage de forma mais dura ao atraso do que o Cliente **sem** cadastro,
uma comparação de dois grupos, não de sete.

Dois refinamentos isolam essa comparação específica:

1. Trocar a faixa de atraso discretizada por `ESTATISTICA_ATRASOSAIDA` contínua, o que
   reduz a interação de 18 para 6 parâmetros (um coeficiente de inclinação por tier, em
   vez de uma célula por combinação tier x faixa).
2. Restringir a comparação aos dois tiers que a hipótese nomeia: `SEM CADASTRO` (sem
   fidelização) e `DIAMANTE` (tier alto de maior volume; `AZUL ONE` e `DIAMANTE UNIQUE`
   ficam de fora por terem menos de 1.200 respostas ao todo).

In [ ]:
# Interação com atraso contínuo, todos os tiers: mesma pergunta do teste anterior, com
# 6 graus de liberdade em vez de 18, porque cada tier ganha só um coeficiente de
# inclinação (efeito por minuto de atraso) em vez de uma célula por faixa.
modelo_aditivo_continuo = smf.logit(
    f"_detrator ~ C({formula_col}) + {COL['atraso_saida']}", data=base_h5
).fit(disp=False)
modelo_interacao_continuo = smf.logit(
    f"_detrator ~ C({formula_col}) * {COL['atraso_saida']}", data=base_h5
).fit(disp=False)

lr_continuo = 2 * (modelo_interacao_continuo.llf - modelo_aditivo_continuo.llf)
gl_continuo = modelo_interacao_continuo.df_model - modelo_aditivo_continuo.df_model
p_continuo = stats.chi2.sf(lr_continuo, gl_continuo)

print("Interação com atraso contínuo (todos os tiers):")
print(f"  estatística = {lr_continuo:.2f}; gl = {gl_continuo}; p = {p_continuo:.3g}")
print(f"  decisão: {'rejeita H0' if p_continuo < ALPHA_H5 else 'não rejeita H0'}")

# Coeficiente de interação por tier: quanto mais positivo, mais a taxa de detração
# desse tier sobe por minuto de atraso em relação ao tier de referência (AZUL FIDELIDADE).
inclinacoes = modelo_interacao_continuo.params.filter(like=f":{COL['atraso_saida']}")
inclinacoes.index = inclinacoes.index.str.replace(
    rf"C\({formula_col}\)\[T\.(.*)\]:{COL['atraso_saida']}", r"\1", regex=True
)
print("\nInclinação adicional por tier, em relação a AZUL FIDELIDADE (referência):")
print(inclinacoes.sort_values())

In [ ]:
# Comparação direta dos dois tiers que a hipótese nomeia: sem fidelização (menor
# tolerância esperada) contra o tier alto de maior volume (maior tolerância esperada).
base_h5_extremos = base_h5[base_h5[formula_col].isin(["SEM CADASTRO", "DIAMANTE"])].copy()
print(f"SEM CADASTRO vs. DIAMANTE: {len(base_h5_extremos):,} respostas")
print(tabela_taxa.loc[["SEM CADASTRO", "DIAMANTE"]].mul(100).round(1))

modelo_aditivo_extremos = smf.logit(
    f"_detrator ~ C({formula_col}) + C(_faixa_atraso)", data=base_h5_extremos
).fit(disp=False)
modelo_interacao_extremos = smf.logit(
    f"_detrator ~ C({formula_col}) * C(_faixa_atraso)", data=base_h5_extremos
).fit(disp=False)
lr_extremos = 2 * (modelo_interacao_extremos.llf - modelo_aditivo_extremos.llf)
gl_extremos = modelo_interacao_extremos.df_model - modelo_aditivo_extremos.df_model
p_extremos = stats.chi2.sf(lr_extremos, gl_extremos)
print(f"\nFaixas discretas: estatística = {lr_extremos:.2f}; gl = {gl_extremos}; "
      f"p = {p_extremos:.3g}")

modelo_aditivo_extremos_cont = smf.logit(
    f"_detrator ~ C({formula_col}) + {COL['atraso_saida']}", data=base_h5_extremos
).fit(disp=False)
modelo_interacao_extremos_cont = smf.logit(
    f"_detrator ~ C({formula_col}) * {COL['atraso_saida']}", data=base_h5_extremos
).fit(disp=False)
lr_extremos_cont = 2 * (modelo_interacao_extremos_cont.llf - modelo_aditivo_extremos_cont.llf)
gl_extremos_cont = modelo_interacao_extremos_cont.df_model - modelo_aditivo_extremos_cont.df_model
p_extremos_cont = stats.chi2.sf(lr_extremos_cont, gl_extremos_cont)
print(f"Atraso contínuo: estatística = {lr_extremos_cont:.2f}; gl = {gl_extremos_cont}; "
      f"p = {p_extremos_cont:.3g}")

### Conclusão revisada

O teste omnibus (18 graus de liberdade, todos os sete tiers) não rejeita H0 no alfa
convencional de 0,05. Mas essa formulação testa uma pergunta mais ampla e menos potente
do que a hipótese propõe. Quando o teste é reformulado para a comparação que a hipótese
realmente afirma, sem fidelização contra o tier alto de maior volume, com atraso contínuo
ou discretizado, e também quando todos os tiers entram com atraso contínuo em vez de
discretizado, a interação é estatisticamente significativa nos quatro casos. A direção é
consistente com a hipótese: em relação a `AZUL FIDELIDADE`, os tiers `DIAMANTE`, `SAFIRA`
e `AZUL ONE` têm inclinação positiva (detração cresce mais rápido por minuto de atraso),
enquanto `SEM CADASTRO` tem inclinação negativa (cresce mais devagar).

A hipótese é considerada confirmada, com a ressalva de que o teste inicialmente
formulado no notebook (tier x faixa discretizada, todos os tiers) não é a forma mais
potente de testá-la, e de que o efeito é mais robusto na comparação entre os tiers
nomeados na hipótese do que no conjunto completo de sete tiers.

### Figura: taxa de detratores por tier de fidelidade e faixa de atraso

Mapa de calor da tabela cruzada acima, para sustentar visualmente o texto da seção 4.2.4.

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5))
sns.heatmap(
    tabela_taxa.mul(100), annot=True, fmt=".1f", cmap="RdYlBu_r",
    linewidths=0.6, linecolor="white", ax=ax,
    cbar_kws={"label": "Taxa de detratores (%)"},
)
ax.set_title("Taxa de detratores por tier de fidelidade e faixa de atraso")
ax.set_xlabel("")
ax.set_ylabel("")
plt.show()

## Hipótese 6: a fragmentação da jornada eleva a detração por exposição, não por desgaste

Testa se jornadas com mais trechos detratam mais porque cada trecho é uma nova chance de
falha operacional (exposição), e não porque a conexão em si cansa o passageiro (desgaste).
O número de trechos é derivado de `BASE_AIRPORTLEG` (contagem de "/" na sequência de
aeroportos): uma jornada direta tem um trecho, uma com uma conexão tem dois, e assim por
diante.

Como número de trechos e duração da viagem estão fortemente correlacionados, a
comparação bruta entre 1 e 2 trechos é confundida por duração: é preciso restringir a
faixa de duração em que os dois grupos coexistem antes de isolar o efeito do trecho em si.

In [ ]:
if checar_colunas(
    [COL["rota"], COL["tempo_voo"], COL["atraso_saida"], COL["cancelamento"], COL["target"]],
    "Hipótese 6",
):
    base_h6 = df.copy()
    base_h6["_detrator"] = is_detrator(base_h6[COL["target"]])
    # BASE_AIRPORTLEG concatena os aeroportos da jornada separados por "/"; o número de
    # trechos é sempre um a menos que o número de aeroportos na sequência.
    base_h6["_trechos"] = base_h6[COL["rota"]].str.count("/")
    base_h6["_trechos_grupo"] = np.select(
        [base_h6["_trechos"] <= 1, base_h6["_trechos"] == 2],
        ["1 trecho", "2 trechos"],
        default="3 trechos ou mais",
    )

    print("Gradiente bruto, sem controlar duração:")
    gradiente_bruto = base_h6.groupby("_trechos_grupo", observed=True)["_detrator"].agg(
        taxa_detratores="mean", n="size"
    )
    print(gradiente_bruto)
    chi2_bruto, p_bruto, gl_bruto, _ = stats.chi2_contingency(
        pd.crosstab(base_h6["_trechos_grupo"], base_h6["_detrator"])
    )
    print(f"qui-quadrado = {chi2_bruto:.1f}; gl = {gl_bruto}; "
          f"p {'< 0,001' if p_bruto < 0.001 else f'= {p_bruto:.3g}'}")

    print("\nMediana de TEMPO_VOO por grupo de trechos (evidência da confusão com duração):")
    print(base_h6.groupby("_trechos_grupo", observed=True)[COL["tempo_voo"]].median())

In [ ]:
# Recorte de 3 a 6 horas: a única faixa de duração em que voos diretos e voos com uma
# conexão coexistem em volume suficiente para comparar. Fora dela, comparar 1 e 2
# trechos seria comparar viagem curta com viagem longa, não trecho com trecho.
recorte_duracao = base_h6[
    base_h6[COL["tempo_voo"]].between(180, 360)
    & base_h6["_trechos_grupo"].isin(["1 trecho", "2 trechos"])
].copy()

print(f"Recorte de 3h a 6h, 1 ou 2 trechos: {len(recorte_duracao):,} respostas")
tabela_recorte = recorte_duracao.groupby("_trechos_grupo", observed=True)["_detrator"].agg(
    taxa_detratores="mean", n="size"
)
print(tabela_recorte)
chi2_recorte, p_recorte, gl_recorte, _ = stats.chi2_contingency(
    pd.crosstab(recorte_duracao["_trechos_grupo"], recorte_duracao["_detrator"])
)
print(f"Todos os voos do recorte: qui-quadrado = {chi2_recorte:.1f}; gl = {gl_recorte}; "
      f"p {'< 0,001' if p_recorte < 0.001 else f'= {p_recorte:.3g}'}")

# Dentro do mesmo recorte de duração, isola só os voos sem nenhuma falha operacional:
# se a diferença entre 1 e 2 trechos desaparecer aqui, o que resta do efeito bruto
# acima é explicado por atraso e cancelamento, não pela conexão em si.
recorte_duracao["_perfeito"] = (recorte_duracao[COL["atraso_saida"]] <= 0) & (
    ~to_bool(recorte_duracao[COL["cancelamento"]])
)
apenas_perfeitos = recorte_duracao[recorte_duracao["_perfeito"]]
tabela_perfeitos = apenas_perfeitos.groupby("_trechos_grupo", observed=True)["_detrator"].agg(
    taxa_detratores="mean", n="size"
)
print("\nApenas voos perfeitos do recorte (sem atraso, sem cancelamento):")
print(tabela_perfeitos)
chi2_perfeitos, p_perfeitos, gl_perfeitos, _ = stats.chi2_contingency(
    pd.crosstab(apenas_perfeitos["_trechos_grupo"], apenas_perfeitos["_detrator"])
)
print(f"qui-quadrado = {chi2_perfeitos:.2f}; gl = {gl_perfeitos}; p = {p_perfeitos:.3g}")

### Modelo ajustado e limite de interpretação

O recorte de voos perfeitos é uma análise de sensibilidade, não uma prova de mediação: atraso e cancelamento podem ser consequências da própria fragmentação. O modelo abaixo estima a associação condicional a duração, atraso e cancelamento no recorte com sobreposição de duração. O resultado deve ser interpretado como evidência compatível — ou não — com exposição operacional, nunca como confirmação causal.

In [ ]:
colunas_h6_modelo = [COL["tempo_voo"], COL["atraso_saida"], COL["cancelamento"]]
if all(coluna in recorte_duracao.columns for coluna in colunas_h6_modelo):
    modelo_h6 = recorte_duracao.dropna(subset=colunas_h6_modelo).copy()
    modelo_h6["_dois_trechos"] = (modelo_h6["_trechos_grupo"] == "2 trechos").astype(int)
    modelo_h6["_cancelado"] = to_bool(modelo_h6[COL["cancelamento"]]).astype(int)
    X_h6 = sm.add_constant(modelo_h6[["_dois_trechos", COL["tempo_voo"], COL["atraso_saida"], "_cancelado"]])
    ajuste_h6 = sm.Logit(modelo_h6["_detrator"], X_h6).fit(disp=False)
    coef = ajuste_h6.params["_dois_trechos"]
    ic = ajuste_h6.conf_int().loc["_dois_trechos"]
    print(f"n ajustado = {len(modelo_h6):,}")
    print(f"OR ajustada (2 vs. 1 trecho) = {np.exp(coef):.3f}")
    print(f"IC 95% = [{np.exp(ic.iloc[0]):.3f}, {np.exp(ic.iloc[1]):.3f}]; p = {ajuste_h6.pvalues['_dois_trechos']:.3g}")
else:
    print("Hipótese 6: faltam colunas para o modelo ajustado.")